# Data Feasibility: California Wildfire Pilot

## Purpose

This notebook checks whether the proposed data sources can support the SmokeWatch Replay project.

The pilot focuses on the 2021 Dixie Fire in California during July 13–August 31, 2021. We will examine whether fire detections, air-quality observations, weather records, and incident information can be cleaned and joined into hourly county-level records.

## Questions

1. Can each source be accessed and downloaded?
2. What fields and time formats does each source contain?
3. Can records be aligned by county and hour?
4. What missing values, duplicates, and timestamp problems must be handled?
5. Is the proposed pilot feasible for the project timeline?

## Data Sources to Validate

| Source | Purpose in the pipeline | Initial validation |
|---|---|---|
| NASA FIRMS | Fire detection events | Check location, date, time, and fire-confidence fields |
| EPA AirData | Hourly PM2.5 observations | Check PM2.5 values, station location, and timestamps |
| NOAA | Weather observations | Check temperature, wind, and observation time |
| CAL FIRE | Fire incident information | Check incident name, dates, counties, and coordinates |

The sources will be evaluated for accessibility, time coverage, geographic coverage, missing values, duplicate records, and compatibility with an hourly county-level join.

## Source 1: NASA FIRMS Active Fire Data

- **Official archive:** https://firms.modaps.eosdis.nasa.gov/download/
- **Area API:** https://firms.modaps.eosdis.nasa.gov/api/area/
- **Role:** Provides satellite-detected active-fire locations and observation times.
- **Initial product:** VIIRS NOAA-20 375 m Standard Processing (`VIIRS_NOAA20_SP`)
- **Temporal fit:** VIIRS NOAA-20 coverage begins on April 1, 2018, so it includes the 2021 Dixie Fire period.
- **Available formats:** CSV, JSON, and Shapefile.
- **Expected fields:** Latitude, longitude, acquisition date, acquisition time, confidence, fire radiative power, satellite, and day/night indicator.
- **Access requirement:** Historical archive access requires an Earthdata login or email verification. Programmatic API access requires a free FIRMS map key.
- **Initial decision:** Begin with the NOAA-20 VIIRS product to avoid duplicate detections from combining multiple satellites.

## Source 2: EPA AirData Hourly PM2.5

- **Official download page:** https://aqs.epa.gov/aqsweb/airdata/download_files.html
- **2021 hourly file:** https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2021.zip
- **Parameter:** PM2.5 FRM/FEM Mass (`88101`)
- **Role:** Provides hourly PM2.5 measurements from outdoor air-quality monitors.
- **Format:** Compressed ZIP containing a CSV file with a header row.
- **Dataset size:** Approximately 8.6 million national records and 67 MB compressed.
- **Important fields:** State Code, County Code, Site Num, latitude, longitude, Date Local, Time Local, Date GMT, Time GMT, Sample Measurement, units, qualifier, and method information.
- **Planned filtering:** Keep California records for Butte, Plumas, Shasta, Lassen, and Tehama counties during July 13–August 31, 2021.
- **Timestamp plan:** Use the GMT date and time fields to align EPA observations with NASA FIRMS timestamps.
- **Initial concern:** A county may have multiple monitoring sites or instruments during the same hour. We must define an aggregation rule, such as calculating the hourly county mean after quality checks.

## Source 3: NOAA Global Hourly Weather Data

- **Official product page:** https://www.ncei.noaa.gov/products/land-based-station/integrated-surface-database
- **Data search:** https://www.ncei.noaa.gov/access/search/datasets/global-hourly/
- **API documentation:** https://www.ncei.noaa.gov/access/search/documentation/data-service/
- **Role:** Provides hourly and sub-hourly weather observations from ground stations.
- **Expected variables:** Temperature, dew point, wind speed, wind direction, wind gust, atmospheric pressure, visibility, and precipitation.
- **Time coverage:** Historical data include the 2021 Dixie Fire period.
- **Planned filtering:** Select weather stations in or near Butte, Plumas, Shasta, Lassen, and Tehama counties for July 13–August 31, 2021.
- **Timestamp plan:** Convert observation timestamps to UTC and round or aggregate them into one-hour windows.
- **Initial concern:** NOAA records are organized by weather station rather than directly by county. Station coordinates must be mapped to counties, and some counties may not contain a suitable station.
- **Initial decision:** Validate station availability before selecting the final weather variables or downloading the complete pilot period.

## Source 4: CAL FIRE Dixie Fire Incident Information

- **Official 2021 incident archive:** https://www.fire.ca.gov/incidents/2021/
- **Official Dixie Fire page:** https://www.fire.ca.gov/incidents/2021/7/13/dixie-fire/
- **Role:** Provides authoritative incident context and metadata for the pilot case study.
- **Available information:** Incident name, location, counties, acreage, containment, status, cause, structures affected, and archived updates.
- **Geographic scope:** Butte, Plumas, Shasta, Lassen, and Tehama counties.
- **Final reported size:** 963,309 acres.
- **Planned use:** Use CAL FIRE information as an incident reference or dimension record rather than as the primary hourly event stream.
- **Initial concern:** CAL FIRE updates are not consistently hourly, and some archived pages show differences in recorded start dates or county coverage as the incident evolved.
- **Initial decision:** Use NASA, EPA, and NOAA for hourly observations. Use CAL FIRE to validate the selected incident, dates, counties, and final incident characteristics.

## Preliminary Feasibility Assessment

| Source | Preliminary status | Main risk | Decision |
|---|---|---|---|
| NASA FIRMS | Feasible | API requests are limited to five-day batches and require a map key | Proceed with one VIIRS product and automate date batches |
| EPA AirData | Feasible | National hourly file is large and may contain multiple monitors per county-hour | Filter by date and county, then define an aggregation rule |
| NOAA Global Hourly | Conditionally feasible | Weather stations may not be available inside every selected county | Validate station coverage before finalizing the geographic join |
| CAL FIRE | Feasible for metadata | Incident updates are not consistently hourly | Use as an incident reference rather than an hourly stream |

## Preliminary Conclusion

The proposed pilot is technically feasible because all four sources cover the 2021 period and provide downloadable historical information. NASA FIRMS, EPA AirData, and NOAA can support the time-based pipeline, while CAL FIRE can provide incident context. The most important unresolved issue is whether suitable NOAA and EPA monitoring stations exist in or near all five selected counties.

## NASA FIRMS NOAA-20 Sample Acquisition

This section tests whether NOAA-20 VIIRS fire-detection data can be retrieved for California during the first five days of the Dixie Fire pilot period.

In [1]:
%pip install -q pandas requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os

import pandas as pd
import requests
from dotenv import load_dotenv

# Locate the repository root whether the notebook starts
# inside the repository or its notebooks folder.
current_folder = Path.cwd()
repo_root = (
    current_folder
    if (current_folder / ".env").exists()
    else current_folder.parent
)

env_path = repo_root / ".env"

assert env_path.exists(), ".env file was not found at the repository root."

load_dotenv(env_path)
firms_map_key = os.getenv("FIRMS_MAP_KEY")

assert firms_map_key, "FIRMS_MAP_KEY is missing from the .env file."

print("Setup complete: FIRMS key loaded securely.")
print("Repository root located successfully.")

Setup complete: FIRMS key loaded securely.
Repository root located successfully.


In [3]:
from io import StringIO

# Request settings: California, five days beginning July 13, 2021
source = "VIIRS_NOAA20_SP"
area = "-124.5,32.5,-114.0,42.1"
day_range = 5
start_date = "2021-07-13"

api_url = (
    "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
    f"{firms_map_key}/{source}/{area}/{day_range}/{start_date}"
)

# Request the data without displaying the URL or MAP_KEY
try:
    response = requests.get(api_url, timeout=90)
except requests.RequestException:
    raise RuntimeError(
        "The NASA FIRMS request encountered a network error."
    ) from None

if response.status_code != 200:
    raise RuntimeError(
        f"NASA FIRMS returned status code {response.status_code}."
    )

firms_sample = pd.read_csv(StringIO(response.text))

required_columns = {"latitude", "longitude", "acq_date", "acq_time"}
missing_columns = required_columns - set(firms_sample.columns)

assert not missing_columns, (
    f"Expected FIRMS columns are missing: {missing_columns}"
)

# Save the retrieved data locally in an ignored folder
output_folder = repo_root / "data" / "raw" / "nasa_firms"
output_folder.mkdir(parents=True, exist_ok=True)

output_path = (
    output_folder
    / "viirs_noaa20_california_2021-07-13_5days.csv"
)

firms_sample.to_csv(output_path, index=False)

print("NASA FIRMS request completed successfully.")
print(f"Rows downloaded: {len(firms_sample):,}")
print(f"Columns received: {len(firms_sample.columns)}")
print(f"Date range: {firms_sample['acq_date'].min()} to "
      f"{firms_sample['acq_date'].max()}")
print(f"Saved locally to: {output_path.relative_to(repo_root)}")

display(firms_sample.head())

NASA FIRMS request completed successfully.
Rows downloaded: 2,956
Columns received: 15
Date range: 2021-07-13 to 2021-07-17
Saved locally to: data/raw/nasa_firms/viirs_noaa20_california_2021-07-13_5days.csv


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,37.79095,-118.89727,296.97,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,284.78,4.54,N,0
1,37.79288,-118.88161,305.53,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,285.27,3.35,N,0
2,37.79301,-118.88929,309.68,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,286.95,3.70,N,0
3,37.79315,-118.89719,309.11,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,286.06,3.70,N,0
4,37.79779,-118.88048,312.54,0.73,0.76,2021-07-13,840,N20,VIIRS,n,2,287.36,5.56,N,0


### Initial NASA FIRMS Data-Quality Check

The downloaded sample is checked for its structure, missing values, duplicate rows, date coverage, and coordinate range before further filtering or transformation.

In [4]:
quality_summary = pd.DataFrame({
    "data_type": firms_sample.dtypes.astype(str),
    "missing_values": firms_sample.isna().sum(),
    "missing_percent": (
        firms_sample.isna().mean() * 100
    ).round(2)
})

print(f"Dataset shape: {firms_sample.shape}")
print(f"Duplicate rows: {firms_sample.duplicated().sum()}")
print(
    f"Unique acquisition dates: "
    f"{firms_sample['acq_date'].nunique()}"
)
print(
    f"Latitude range: {firms_sample['latitude'].min():.4f} "
    f"to {firms_sample['latitude'].max():.4f}"
)
print(
    f"Longitude range: {firms_sample['longitude'].min():.4f} "
    f"to {firms_sample['longitude'].max():.4f}"
)

display(quality_summary)

Dataset shape: (2956, 15)
Duplicate rows: 0
Unique acquisition dates: 5
Latitude range: 32.5014 to 41.4527
Longitude range: -123.6966 to -114.1916


,data_type,missing_values,missing_percent
latitude,float64,0,0.0
longitude,float64,0,0.0
bright_ti4,float64,0,0.0
scan,float64,0,0.0
track,float64,0,0.0
acq_date,object,0,0.0
acq_time,int64,0,0.0
satellite,object,0,0.0
instrument,object,0,0.0
confidence,object,0,0.0


### Create a Standard UTC Event Timestamp

NASA stores the acquisition date and time in separate columns. These fields are combined into one UTC timestamp so fire detections can later be grouped into hourly windows and joined with EPA and NOAA observations.

In [5]:
firms_clean = firms_sample.copy()

# Restore acquisition time to four digits.
# For example: 840 becomes 0840 and 5 becomes 0005.
firms_clean["acq_time_text"] = (
    firms_clean["acq_time"]
    .astype(str)
    .str.zfill(4)
)

# Combine the acquisition date and time as a UTC timestamp.
firms_clean["event_timestamp_utc"] = pd.to_datetime(
    firms_clean["acq_date"].astype(str)
    + " "
    + firms_clean["acq_time_text"],
    format="%Y-%m-%d %H%M",
    utc=True,
    errors="coerce"
)

invalid_timestamps = firms_clean[
    "event_timestamp_utc"
].isna().sum()

print(f"Invalid timestamps: {invalid_timestamps}")
print(
    "Earliest timestamp:",
    firms_clean["event_timestamp_utc"].min()
)
print(
    "Latest timestamp:",
    firms_clean["event_timestamp_utc"].max()
)

display(
    firms_clean[
        [
            "acq_date",
            "acq_time",
            "acq_time_text",
            "event_timestamp_utc"
        ]
    ].head()
)

Invalid timestamps: 0
Earliest timestamp: 2021-07-13 08:40:00+00:00
Latest timestamp: 2021-07-17 22:09:00+00:00


,acq_date,acq_time,acq_time_text,event_timestamp_utc
0,2021-07-13,840,0840,2021-07-13 08:40:00+00:00
1,2021-07-13,840,0840,2021-07-13 08:40:00+00:00
2,2021-07-13,840,0840,2021-07-13 08:40:00+00:00
3,2021-07-13,840,0840,2021-07-13 08:40:00+00:00
4,2021-07-13,840,0840,2021-07-13 08:40:00+00:00


### Inspect Fire-Detection Categories

The confidence, hotspot type, and day/night fields are examined before selecting records for the wildfire pilot. This avoids removing records without first understanding their meaning and frequency.

In [6]:
confidence_counts = (
    firms_clean["confidence"]
    .value_counts(dropna=False)
    .rename_axis("confidence")
    .reset_index(name="record_count")
)

type_counts = (
    firms_clean["type"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("type")
    .reset_index(name="record_count")
)

daynight_counts = (
    firms_clean["daynight"]
    .value_counts(dropna=False)
    .rename_axis("daynight")
    .reset_index(name="record_count")
)

print("Confidence categories:")
display(confidence_counts)

print("Hotspot types:")
display(type_counts)

print("Day/night categories:")
display(daynight_counts)

Confidence categories:


,confidence,record_count
0,n,2330
1,h,354
2,l,272


Hotspot types:


,type,record_count
0,0,2860
1,2,96


Day/night categories:


,daynight,record_count
0,N,1870
1,D,1086


### NASA FIRMS Filtering Decision

For the primary wildfire analysis, only presumed vegetation fires (`type = 0`) with nominal (`n`) or high (`h`) confidence are retained. Other static land sources and low-confidence detections are excluded to reduce unrelated heat sources and potential false positives. Both daytime and nighttime detections are retained.

In [7]:
firms_filtered = firms_clean[
    (firms_clean["type"] == 0)
    & (firms_clean["confidence"].isin(["n", "h"]))
].copy()

# Create the hourly key required for later integration.
firms_filtered["event_hour_utc"] = (
    firms_filtered["event_timestamp_utc"].dt.floor("h")
)

records_removed = len(firms_clean) - len(firms_filtered)
percent_retained = (
    len(firms_filtered) / len(firms_clean) * 100
)

print(f"Original records: {len(firms_clean):,}")
print(f"Filtered records: {len(firms_filtered):,}")
print(f"Records removed: {records_removed:,}")
print(f"Percentage retained: {percent_retained:.2f}%")
print(
    f"Unique hourly windows: "
    f"{firms_filtered['event_hour_utc'].nunique()}"
)

display(
    firms_filtered[
        [
            "event_timestamp_utc",
            "event_hour_utc",
            "latitude",
            "longitude",
            "confidence",
            "type",
            "frp",
            "daynight"
        ]
    ].head()
)

Original records: 2,956
Filtered records: 2,591
Records removed: 365
Percentage retained: 87.65%
Unique hourly windows: 17


,event_timestamp_utc,event_hour_utc,latitude,longitude,confidence,type,frp,daynight
0,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79095,-118.89727,n,0,4.54,N
1,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79288,-118.88161,n,0,3.35,N
2,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79301,-118.88929,n,0,3.70,N
3,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79315,-118.89719,n,0,3.70,N
4,2021-07-13 08:40:00+00:00,2021-07-13 08:00:00+00:00,37.79779,-118.88048,n,0,5.56,N


In [8]:
%pip install -q geopandas pyogrio

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Note: you may need to restart the kernel to use updated packages.


In [9]:
import geopandas as gpd
import pyogrio

print("GeoPandas version:", gpd.__version__)
print("Pyogrio version:", pyogrio.__version__)
print("Geographic packages imported successfully.")

GeoPandas version: 1.2.0
Pyogrio version: 0.13.0
Geographic packages imported successfully.


### County Boundary Reference Data

Official 2021 U.S. Census cartographic boundary data is used to map each NASA FIRMS latitude and longitude to a California county. Only Butte, Plumas, Shasta, Lassen, and Tehama counties are retained for the pilot.

In [10]:
county_boundary_url = (
    "https://www2.census.gov/geo/tiger/GENZ2021/shp/"
    "cb_2021_us_county_500k.zip"
)

reference_folder = repo_root / "data" / "raw" / "reference"
reference_folder.mkdir(parents=True, exist_ok=True)

county_zip_path = (
    reference_folder / "cb_2021_us_county_500k.zip"
)

# Download once and reuse the local copy in later notebook runs.
if not county_zip_path.exists():
    boundary_response = requests.get(
        county_boundary_url,
        timeout=120
    )

    if boundary_response.status_code != 200:
        raise RuntimeError(
            "The Census county-boundary download failed with "
            f"status code {boundary_response.status_code}."
        )

    county_zip_path.write_bytes(boundary_response.content)
    print("County-boundary file downloaded.")
else:
    print("Using the existing local county-boundary file.")

county_boundaries = gpd.read_file(
    f"zip://{county_zip_path}"
)

pilot_county_names = [
    "Butte",
    "Plumas",
    "Shasta",
    "Lassen",
    "Tehama"
]

pilot_counties = county_boundaries[
    (county_boundaries["STATEFP"] == "06")
    & (county_boundaries["NAME"].isin(pilot_county_names))
].copy()

assert set(pilot_counties["NAME"]) == set(
    pilot_county_names
), "One or more pilot counties were not found."

print(f"Boundary CRS: {pilot_counties.crs}")
print(f"Pilot counties found: {len(pilot_counties)}")

display(
    pilot_counties[
        ["STATEFP", "COUNTYFP", "GEOID", "NAME"]
    ].sort_values("NAME")
)

County-boundary file downloaded.
Boundary CRS: EPSG:4269
Pilot counties found: 5


,STATEFP,COUNTYFP,GEOID,NAME
551,06,007,06007,Butte
2573,06,035,06035,Lassen
94,06,063,06063,Plumas
807,06,089,06089,Shasta
802,06,103,06103,Tehama


### Assign Fire Detections to Pilot Counties

Each filtered NASA FIRMS coordinate is converted into a geographic point and spatially joined with the Census county boundaries. This adds an exact county name and county GEOID to each matched detection.

In [11]:
# Convert FIRMS latitude and longitude into geographic points.
firms_points = gpd.GeoDataFrame(
    firms_filtered.copy(),
    geometry=gpd.points_from_xy(
        firms_filtered["longitude"],
        firms_filtered["latitude"]
    ),
    crs="EPSG:4326"
)

# Match the coordinate reference system used by the boundaries.
firms_points = firms_points.to_crs(
    pilot_counties.crs
)

# Assign each point to one of the five pilot counties.
firms_pilot = gpd.sjoin(
    firms_points,
    pilot_counties[
        ["GEOID", "NAME", "geometry"]
    ],
    how="inner",
    predicate="within"
)

firms_pilot = (
    firms_pilot
    .drop(columns=["index_right"])
    .rename(
        columns={
            "GEOID": "county_geoid",
            "NAME": "county"
        }
    )
)

county_counts = (
    firms_pilot["county"]
    .value_counts()
    .rename_axis("county")
    .reset_index(name="fire_detection_count")
)

print(
    f"Statewide filtered records: "
    f"{len(firms_filtered):,}"
)
print(
    f"Records inside pilot counties: "
    f"{len(firms_pilot):,}"
)
print(
    f"Pilot counties with detections: "
    f"{firms_pilot['county'].nunique()}"
)

display(county_counts)

Statewide filtered records: 2,591
Records inside pilot counties: 1,209
Pilot counties with detections: 4


,county,fire_detection_count
0,Plumas,1085
1,Lassen,113
2,Butte,6
3,Tehama,5


### Feasibility Finding: County and Date Are Not Sufficient

The first five-day sample contains 1,209 fire detections within the five pilot counties, but the distribution suggests that other fires are also present. Therefore, county and date alone cannot identify Dixie Fire observations. An official CAL FIRE incident perimeter will be used as an additional spatial filter.

The available CAL FIRE perimeter represents the completed fire boundary rather than its exact daily progression. It is therefore suitable as an incident-area mask, but this limitation must be documented.

In [12]:
calfire_query_url = (
    "https://services.arcgis.com/V6ZHFr6zdgNZuVG0/"
    "ArcGIS/rest/services/California_Fire_Perimeters/"
    "FeatureServer/0/query"
)

query_parameters = {
    "where": "YEAR_ = 2021 AND FIRE_NAME = 'DIXIE'",
    "outFields": (
        "OBJECTID,YEAR_,FIRE_NAME,GIS_ACRES,"
        "ALARM_DATE,CONT_DATE"
    ),
    "returnGeometry": "true",
    "outSR": "4326",
    "f": "geojson"
}

perimeter_response = requests.get(
    calfire_query_url,
    params=query_parameters,
    timeout=120
)

if perimeter_response.status_code != 200:
    raise RuntimeError(
        "CAL FIRE perimeter request failed with status "
        f"{perimeter_response.status_code}."
    )

perimeter_data = perimeter_response.json()

if "error" in perimeter_data:
    raise RuntimeError(
        f"CAL FIRE API error: {perimeter_data['error']}"
    )

dixie_perimeter = gpd.GeoDataFrame.from_features(
    perimeter_data["features"],
    crs="EPSG:4326"
)

assert not dixie_perimeter.empty, (
    "No 2021 Dixie Fire perimeter was returned."
)

print(
    f"Dixie perimeter records returned: "
    f"{len(dixie_perimeter)}"
)

display(
    dixie_perimeter[
        [
            "YEAR_",
            "FIRE_NAME",
            "GIS_ACRES",
            "ALARM_DATE",
            "CONT_DATE"
        ]
    ]
)

Dixie perimeter records returned: 1


,YEAR_,FIRE_NAME,GIS_ACRES,ALARM_DATE,CONT_DATE
0,2021,DIXIE,963405.4,1626134400000,1635120000000


### Isolate the Dixie Fire Incident Area

The final CAL FIRE Dixie Fire perimeter is used as an incident-area mask. A one-kilometer buffer is added to account for VIIRS pixel size and small differences between satellite coordinates and the generalized perimeter boundary.

This is a spatial approximation. Because the final perimeter includes areas burned later in the incident, it does not represent the fire's exact daily progression.

In [13]:
# Convert CAL FIRE date values from milliseconds to readable UTC dates.
for date_column in ["ALARM_DATE", "CONT_DATE"]:
    dixie_perimeter[date_column] = pd.to_datetime(
        dixie_perimeter[date_column],
        unit="ms",
        utc=True,
        errors="coerce"
    )

# Save the original CAL FIRE perimeter locally.
dixie_perimeter_path = (
    reference_folder
    / "calfire_dixie_2021_final_perimeter.geojson"
)

dixie_perimeter.to_file(
    dixie_perimeter_path,
    driver="GeoJSON"
)

# Use California Albers so the buffer can be measured in meters.
projected_crs = "EPSG:3310"

dixie_projected = dixie_perimeter.to_crs(projected_crs)
pilot_points_projected = firms_pilot.to_crs(projected_crs)

# Combine the perimeter geometry and add a 1-kilometer buffer.
dixie_buffered_geometry = (
    dixie_projected.geometry.union_all().buffer(1000)
)

dixie_mask = gpd.GeoDataFrame(
    {"incident": ["Dixie Fire"]},
    geometry=[dixie_buffered_geometry],
    crs=projected_crs
)

# Retain only detections inside the buffered Dixie perimeter.
firms_dixie = gpd.sjoin(
    pilot_points_projected,
    dixie_mask,
    how="inner",
    predicate="within"
).drop(columns=["index_right"])

dixie_county_counts = (
    firms_dixie["county"]
    .value_counts()
    .rename_axis("county")
    .reset_index(name="fire_detection_count")
)

print(
    f"Pilot-county records before perimeter filter: "
    f"{len(firms_pilot):,}"
)
print(
    f"Dixie-area records after perimeter filter: "
    f"{len(firms_dixie):,}"
)
print(
    f"Counties represented: "
    f"{firms_dixie['county'].nunique()}"
)
print(
    "Perimeter dates:",
    dixie_perimeter["ALARM_DATE"].min(),
    "to",
    dixie_perimeter["CONT_DATE"].max()
)

display(dixie_county_counts)

Pilot-county records before perimeter filter: 1,209
Dixie-area records after perimeter filter: 976
Counties represented: 3
Perimeter dates: 2021-07-13 00:00:00+00:00 to 2021-10-25 00:00:00+00:00


,county,fire_detection_count
0,Plumas,964
1,Butte,6
2,Lassen,6


### Validate the Daily Geographic Pattern

The perimeter-filtered detections are summarized by date and county. This check helps determine whether the records show a reasonable temporal progression rather than an unexplained concentration from another incident.

In [14]:
daily_county_counts = (
    firms_dixie
    .groupby(["acq_date", "county"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=pilot_county_names, fill_value=0)
)

daily_county_counts["daily_total"] = (
    daily_county_counts.sum(axis=1)
)

print("Dixie-area detections by date and county:")
display(daily_county_counts)

print(
    "Total filtered Dixie-area records:",
    f"{daily_county_counts['daily_total'].sum():,}"
)

Dixie-area detections by date and county:


county,Butte,Plumas,Shasta,Lassen,Tehama,daily_total
acq_date,,,,,,
2021-07-13,0,46,0,0,0,46
2021-07-14,0,98,0,0,0,98
2021-07-15,4,171,0,6,0,181
2021-07-16,2,268,0,0,0,270
2021-07-17,0,381,0,0,0,381


Total filtered Dixie-area records: 976


### Create the Hourly County-Level Fire Dataset

The incident-area detections are aggregated by county and UTC hour. A complete hourly grid is created for all five pilot counties so that hours without a fire detection are represented with a count of zero rather than being absent from the dataset.

Mean and maximum FRP remain missing when no fire was detected because an average or maximum does not exist for an empty group.

In [15]:
observed_hourly_fire = (
    firms_dixie
    .groupby(
        ["event_hour_utc", "county"],
        observed=True
    )
    .agg(
        fire_detection_count=("latitude", "size"),
        total_frp_mw=("frp", "sum"),
        mean_frp_mw=("frp", "mean"),
        max_frp_mw=("frp", "max"),
        high_confidence_count=(
            "confidence",
            lambda values: (values == "h").sum()
        ),
        night_detection_count=(
            "daynight",
            lambda values: (values == "N").sum()
        )
    )
    .reset_index()
)

# Create every UTC hour for all five counties.
all_hours = pd.date_range(
    start="2021-07-13 00:00:00",
    end="2021-07-17 23:00:00",
    freq="h",
    tz="UTC"
)

complete_hourly_grid = (
    pd.MultiIndex.from_product(
        [all_hours, pilot_county_names],
        names=["event_hour_utc", "county"]
    )
    .to_frame(index=False)
)

hourly_fire_sample = complete_hourly_grid.merge(
    observed_hourly_fire,
    on=["event_hour_utc", "county"],
    how="left"
)

count_columns = [
    "fire_detection_count",
    "high_confidence_count",
    "night_detection_count"
]

hourly_fire_sample[count_columns] = (
    hourly_fire_sample[count_columns]
    .fillna(0)
    .astype(int)
)

hourly_fire_sample["total_frp_mw"] = (
    hourly_fire_sample["total_frp_mw"].fillna(0)
)

# Save the cleaned hourly sample locally.
interim_folder = (
    repo_root / "data" / "interim" / "nasa_firms"
)
interim_folder.mkdir(parents=True, exist_ok=True)

hourly_output_path = (
    interim_folder
    / "dixie_fire_hourly_county_sample_2021-07-13_17.csv"
)

hourly_fire_sample.to_csv(
    hourly_output_path,
    index=False
)

print(f"Hourly county rows: {len(hourly_fire_sample):,}")
print(
    "Rows with one or more detections:",
    (
        hourly_fire_sample["fire_detection_count"] > 0
    ).sum()
)
print(
    "Total detections preserved:",
    hourly_fire_sample["fire_detection_count"].sum()
)
print(
    "Saved locally to:",
    hourly_output_path.relative_to(repo_root)
)

display(
    hourly_fire_sample[
        hourly_fire_sample["fire_detection_count"] > 0
    ].head(10)
)

Hourly county rows: 600
Rows with one or more detections: 19
Total detections preserved: 976
Saved locally to: data/interim/nasa_firms/dixie_fire_hourly_county_sample_2021-07-13_17.csv


,event_hour_utc,county,fire_detection_count,total_frp_mw,mean_frp_mw,max_frp_mw,high_confidence_count,night_detection_count
51,2021-07-13 10:00:00+00:00,Plumas,16,47.98,2.998750,10.45,0,16
101,2021-07-13 20:00:00+00:00,Plumas,20,541.19,27.059500,97.71,6,0
106,2021-07-13 21:00:00+00:00,Plumas,10,533.12,53.312000,146.02,5,0
171,2021-07-14 10:00:00+00:00,Plumas,44,264.24,6.005455,34.91,1,44
216,2021-07-14 19:00:00+00:00,Plumas,7,192.69,27.527143,51.59,2,0
226,2021-07-14 21:00:00+00:00,Plumas,47,3735.42,79.477021,558.40,18,0
285,2021-07-15 09:00:00+00:00,Butte,4,6.15,1.537500,2.40,0,4
286,2021-07-15 09:00:00+00:00,Plumas,96,1107.82,11.539792,78.11,1,96
288,2021-07-15 09:00:00+00:00,Lassen,1,35.54,35.540000,35.54,0,1
346,2021-07-15 21:00:00+00:00,Plumas,75,5351.45,71.352667,263.74,34,0


## NASA FIRMS Feasibility Result

NASA FIRMS is feasible for the SmokeWatch Replay project.

The five-day NOAA-20 sample contained 2,956 California fire detections. After removing low-confidence detections and non-vegetation heat sources, 2,591 records remained. A county spatial join identified 1,209 records within the five pilot counties.

County and date filtering alone was not sufficient to isolate the Dixie Fire because other wildfire activity was present in the region. Applying the official CAL FIRE Dixie Fire perimeter with a one-kilometer buffer reduced the sample to 976 incident-area detections.

All acquisition dates and times were converted successfully into UTC timestamps. The detections were then aggregated into a complete dataset of 600 county-hour rows, representing five counties across 120 hours. Hours without detections were explicitly recorded with a count of zero.

### Limitations

- NASA satellite detections occur during satellite passes rather than continuously every hour.
- The CAL FIRE boundary represents the final fire perimeter, not its exact daily progression.
- The one-kilometer perimeter buffer is an analytical assumption used to accommodate satellite pixel size and boundary uncertainty.
- The five-day sample establishes feasibility but does not yet represent the complete July 13–August 31 pilot period.

### Decision

Proceed with NASA FIRMS as the fire-detection source. The full pilot period can be retrieved in five-day API batches and processed using the validated cleaning, spatial-filtering, and hourly-aggregation steps.